# 01 — Exploratory data analysis

**Question:** do companies that get more Wikipedia attention before their IPO have bigger first-day or 30-day returns?

This notebook only *looks* at the data. It doesn't test anything formally. The goal is to understand the data before trusting any model.

**Publishing rules (read first):**
- Cells and figures marked **LOCAL ONLY — do not publish** show individual companies. They exist to help you understand the data and must never leave this machine.
- Everything else is an aggregate. Use `publishable()` before showing groups: any group with fewer than 15 firms gets hidden.
- The return data comes from a licensed vendor. Here it's only an outcome to study, never a check on other data.

## Settings

`INCLUDE_NO_ARTICLE` is the one switch every section respects (later it becomes a Streamlit toggle).

- `True` (**main spec**): firms whose Wikipedia article didn't exist before the IPO stay in, counted as **zero attention**. The logic: no article *is* low attention.
- `False` (**robustness check**): drop those 48 firms. If the results only hold one way, that's important to know.

In [ ]:
INCLUDE_NO_ARTICLE = True   # main spec; set False for the robustness check
MIN_N = 15                  # publish threshold for any group

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.stats import spearmanr
pd.set_option('display.width', 140)
ROOT = '..'

## Helpers

`publishable(df, n_col)` blanks out any row whose group has n < 15, so you can't accidentally publish a tiny group. `sample(df, include)` applies the switch in one place, so every section filters the same way.

In [ ]:
def publishable(df, n_col='n', keys=('include_no_article','group','year')):
    """Return a copy with groups of n < MIN_N suppressed (values -> NaN, flagged)."""
    out = df.copy()
    small = out[n_col] < MIN_N
    for c in out.columns:
        if c != n_col and c not in keys:
            out[c] = out[c].astype(object).where(~small, 'suppressed (n<15)')
    return out

def sample(df, include=None):
    include = INCLUDE_NO_ARTICLE if include is None else include
    return df if include else df[df.has_pre_ipo_article]

## 1. Load and sanity-check

Before analysing anything, check the data is what we think it is:
- **Nulls:** missing values silently drop rows from calculations.
- **Duplicates:** one company counted twice would bias everything.
- **Ranges:** e.g. negative pageviews would mean a bug.
- **Outlier returns** (>+500% or <−90%): real IPOs do this sometimes, but so do data errors (wrong price, stock split). We **flag** them for a human to look at; we don't delete them, because deleting extreme-but-real values hides the most interesting cases.

We keep the 306 firms with views plus a day-30 return (US-IPO set only). For **20 firms the vendor's first-day close equals the offer price exactly**, which is almost certainly a placeholder bar and not a real close. Their first-day return is set to **missing** (flag `pop_close_eq_offer`), so day-1 results use n=286. Their day-30 return is kept.

For regressions later there are `*_capped` columns (clipped at the 1st/99th percentile) and `*_log` columns (log(1+return)). The 3 extreme day-30 firms stay in.

In [ ]:
t = pd.read_parquet(f'{ROOT}/data/processed/analysis_table.parquet')
df = t[t.usable_day30].copy()   # 306 firms; pop is missing for the 20 close==offer firms
df['log_pre30'] = np.log1p(df.views_pre30)
df['outlier_flag'] = (df[['pop','ret30']].gt(5).any(axis=1)) | (df[['pop','ret30']].lt(-0.9).any(axis=1))
print('rows', len(df), '| duplicate CIKs', df.cik.duplicated().sum(), '| duplicate titles', df.title.duplicated().sum())
print('no pre-IPO article:', (~df.has_pre_ipo_article).sum())
print('nulls in key columns:'); print(df[['views_pre30','pop','ret30','sector','year','offer_price']].isna().sum().to_string())
print(df[['views_pre30','views_post30','pop','ret30','offer_price']].describe().round(3))
print('negative views:', (df[['views_pre30','views_post30']] < 0).sum().sum())
print('outlier-flagged firms:', df.outlier_flag.sum(), '| pop set missing (close==offer):', df.pop_close_eq_offer.sum())
print('median day-30 return (exact):', df.ret30.median(), '| median first-day:', df['pop'].median())

**LOCAL ONLY — do not publish.** The flagged firms, so you can check each one by hand (look for splits or a wrong offer price).

In [ ]:
# LOCAL ONLY — do not publish
df.loc[df.outlier_flag, ['legal_name','tickers','year','offer_price','pop','ret30','returns_status']]

## 2. Attention curve

This is the median daily pageviews for each day from 120 days before the IPO to 60 days after. We use the **median**, not the mean, because a few famous companies get millions of views and would swamp the average. The spike at day 0 shows how much the IPO itself drives attention. The level before day 0 is what our hypothesis is about.

In [ ]:
def curves(d):
    rows = []
    for _, r in d.iterrows():
        v = pd.read_csv(f'{ROOT}/data/processed/analysis_views/{r.cik}.csv')
        v['sector'] = r.sector; rows.append(v)
    return pd.concat(rows)
d = sample(df)
cv = curves(d)
overall = cv.groupby('rel_day').views.median()
fig, ax = plt.subplots(figsize=(9,4))
ax.plot(overall.index, overall.values)
ax.axvline(0, ls='--', c='grey'); ax.set_yscale('symlog')
ax.set(title=f'Median daily views around IPO (n={len(d)}, INCLUDE_NO_ARTICLE={INCLUDE_NO_ARTICLE})', xlabel='days from first trade', ylabel='median views')
plt.show()
print('median views: day -30..-1 avg', overall.loc[-30:-1].mean().round(1), '| day 0', overall.loc[0], '| day 1..30 avg', overall.loc[1:30].mean().round(1))

By sector. Only sectors with n ≥ 15 are drawn; smaller ones go into Other, which gets suppressed if it's still under 15.

In [ ]:
def sector_group(d):
    vc = d.sector.value_counts()
    return d.sector.where(d.sector.map(vc) >= MIN_N, 'Other')
d = sample(df).assign(sec=lambda x: sector_group(x))
cv = curves(d.assign(sector=d.sec))
fig, ax = plt.subplots(figsize=(9,5))
for s, g in cv.groupby('sector'):
    n = (d.sec == s).sum()
    if n < MIN_N: continue
    m = g.groupby('rel_day').views.median()
    ax.plot(m.index, m.values, label=f'{s} (n={n})')
ax.axvline(0, ls='--', c='grey'); ax.set_yscale('symlog'); ax.legend(fontsize=8)
ax.set(title='Median daily views by sector', xlabel='days from first trade', ylabel='median views'); plt.show()

## 3. Distributions

Pageviews are extremely skewed: most firms get a few views a day and a handful get thousands. Taking `log(views + 1)` spreads them out so the chart is readable. The `+1` keeps zero-view firms in, since log(0) is undefined. Returns are shown as raw percentages.

In [ ]:
d = sample(df)
fig, axs = plt.subplots(1, 3, figsize=(14,4))
axs[0].hist(d.log_pre30, bins=30); axs[0].set(title='log(pre-IPO 30-day views + 1)')
axs[1].hist(d['pop'].dropna().clip(-1, 3), bins=40); axs[1].set(title='First-day return (clipped at 300%)')
axs[2].hist(d.ret30.clip(-1, 3), bins=40); axs[2].set(title='Day-30 return (clipped at 300%)')
plt.suptitle(f'n={len(d)}'); plt.tight_layout(); plt.show()
print(d[['log_pre30','pop','ret30']].describe().round(3))
print('share with zero pre-IPO 30d views:', (d.views_pre30 == 0).mean().round(3))

## 4. First-day vs day-30 return — **LOCAL ONLY — do not publish**

Each dot is one company, so this figure shows individual firms' returns and stays on this machine. It shows whether day-1 winners keep winning over a month or give it back.

In [ ]:
# LOCAL ONLY — do not publish (per-firm points)
d = sample(df)
fig, ax = plt.subplots(figsize=(6,6))
ax.scatter(d['pop'].clip(-1,3), d.ret30.clip(-1,3), s=10, alpha=.6)
ax.axhline(0, c='grey', lw=.5); ax.axvline(0, c='grey', lw=.5)
ax.set(xlabel='first-day return', ylabel='day-30 return', title='LOCAL ONLY — do not publish')
plt.show()
d1 = d.dropna(subset=['pop']); r, p = spearmanr(d1['pop'], d1.ret30); print(f'Spearman first-day vs day-30: rho={r:.3f}, p={p:.3g}, n={len(d1)}')

## 5. First look: does pre-IPO attention line up with returns?

We use the **Spearman** correlation because it uses ranks, not raw values: "do higher-attention firms tend to have higher returns?" That makes it robust to the giant outliers in both views and returns. rho runs from −1 to +1, and around ±0.1 is weak.

**Caveat:** this is a first look, not a finding. It doesn't control for deal size, year, or the hot 2021 market. We run it under **both** settings because a result that flips between them is fragile.

In [ ]:
def spear(d, y):
    d = d.dropna(subset=[y])
    if len(d) < 3: return np.nan, np.nan
    r, p = spearmanr(d.log_pre30, d[y]); return r, p
rows = []
for inc in (True, False):
    d = sample(df, inc).assign(sec=lambda x: sector_group(x))
    for grp, g in [('All', d)] + list(d.groupby('sec')):
        r1, p1 = spear(g, 'pop'); r30, p30 = spear(g, 'ret30')
        rows.append(dict(include_no_article=inc, group=grp, n=len(g), n_day1=int(g['pop'].notna().sum()), rho_day1=round(r1,3), p_day1=round(p1,4), rho_day30=round(r30,3), p_day30=round(p30,4)))
res = pd.DataFrame(rows)
publishable(res.set_index(['include_no_article','group']).reset_index(), 'n')

Note: sector p-values are uncorrected. With 7 groups × 2 outcomes, about 1 in 20 tests will look "significant" by chance. Treat any single sector result as a lead, not a finding.

## 6. Year composition

The IPO market was very uneven: 2021 was a boom, 2022–23 nearly dried up. If one year dominates the sample, results may describe that year, not IPOs in general.

In [ ]:
d = sample(df)
yc = d.groupby('year').agg(n=('cik','size'), median_day1=('pop','median'), median_day30=('ret30','median'), median_pre30_views=('views_pre30','median')).round(3).reset_index()
display(publishable(yc, 'n'))
yc.set_index('year').n.plot.bar(title='Firms per year'); plt.show()

## 7. Exploratory — multiple comparisons

**Read this carefully.** This is a scan of many variables at once. Test 40 things and about 2 will look "significant" by pure chance. So each p-value is adjusted with **Benjamini–Hochberg** (`q`): q < 0.10 means "fewer than ~10% of the hits at this level are expected to be flukes". Anything that survives is a *lead for a proper model*, not a result. Everything here is aggregate (correlations), so nothing identifies a firm.

- Attention variables are scanned under **both** `INCLUDE_NO_ARTICLE` settings. Deal variables don't depend on the switch, so they use all firms.
- **Reverse direction:** IPO-day and post-IPO views happen *after* trading starts, so they can't predict the pop. Checking them tests whether a big pop *drives* attention afterwards.
- **Article age at IPO** uses the article's first revision. Pages that were moved can show a later date. Negative age means the article was created after the IPO.
- Binary variables (NYSE vs Nasdaq, foreign issuer) use Spearman on a 0/1 code, which is equivalent to a rank-based group comparison.
- Price vs filing range isn't available: we never parsed the preliminary price range.

In [ ]:
def bh(p):
    p = np.asarray(p, float); n = len(p); o = np.argsort(p)
    q = np.empty(n); q[o] = np.minimum.accumulate((p[o] * n / np.arange(1, n+1))[::-1])[::-1]
    return np.minimum(q, 1)
s = df.copy()
s['log_proceeds'] = np.log(s.gross_proceeds.astype(float))
s['log_offer_price'] = np.log(s.offer_price.astype(float))
s['is_nyse'] = (s.exchange == 'NYSE').astype(int)
s['foreign'] = s.foreign_issuer.astype(int)
s['year_num'] = s.year.astype(int)
s['log_trend'] = np.log(s.attn_trend).where(s.views_pre120 > 0)   # undefined with zero pre-IPO views (would otherwise be an artificial ratio of 3)
s['log_pre7'] = np.log1p(s.views_pre7); s['log_day0'] = np.log1p(s.views_day0); s['log_post30'] = np.log1p(s.views_post30)
DEAL = ['log_proceeds','log_offer_price','is_nyse','foreign','year_num','article_age_days']
ATTN = ['log_pre30','log_pre7','log_trend']
REV  = ['log_day0','log_post30']
tests = []
def add(d, x, y, setting, direction):
    g = d[[x, y]].dropna()
    r, p = spearmanr(g[x], g[y]) if len(g) >= MIN_N else (np.nan, np.nan)
    tests.append(dict(variable=x, outcome=y, setting=setting, direction=direction, n=len(g), rho=r, p=p))
for y in ('pop','ret30'):
    for x in DEAL: add(s, x, y, 'all firms', 'predictor')
    for inc in (True, False):
        for x in ATTN: add(sample(s, inc), x, y, f'INCLUDE_NO_ARTICLE={inc}', 'predictor')
for inc in (True, False):
    for x in REV: add(sample(s, inc), x, 'pop', f'INCLUDE_NO_ARTICLE={inc}', 'reverse: pop -> later views')
scan = pd.DataFrame(tests)
scan['q_BH'] = bh(scan.p)
scan = scan.round({'rho':3,'p':4,'q_BH':4}).sort_values('p')
print(len(scan), 'tests; BH across all of them')
publishable(scan, 'n')

In [ ]:
print('Survive q < 0.10:'); scan[scan.q_BH < 0.10]

## Takeaways and next decisions

See the printed numbers above. Open questions are listed in the project notes: how to handle outliers, whether to control for year and deal size, and whether `views_pre30` is the right attention measure.